# Session 9 lab · Motion & video

The code behind the slides. Each cell shows **one idea** in a few lines:

1. run it and look at the result;
2. change a value marked **`· try`** (the values after it are good ones to try) and run it again;
3. read *What to notice* under the cell.

| Part | Ideas |
|---|---|
| **A · Optical flow** | flow on a known shift · flow between two real frames · u = −Iₜ ÷ Iₓ on one row · a bar behind a round hole |
| **B · Lucas–Kanade** | M, its eigenvalues and (u, v) in one window · pyramid Lucas–Kanade on the cyclist |
| **C · Tracking** | frame difference · linking boxes by IoU by hand · YOLO + a tracker, counting at a line |
| **D · Putting it together** | time every stage on your machine · run the detector every N frames |

It runs on free Colab. The first run installs Ultralytics (YOLO; licence AGPL-3.0 — fine for learning) and downloads YOLOv8n (6 MB) and the clip (6 MB): 77 frames of the slides' street, Raysonho, Yonge & Dundas, Toronto (CC0).

In [ ]:
# Setup — run this cell first
import importlib.util, subprocess, sys
if importlib.util.find_spec('ultralytics') is None:                     # Colab: install YOLO once
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics'], check=True)
import numpy as np, matplotlib.pyplot as plt, cv2, urllib.request, os, time
from ultralytics import YOLO

def fetch(name):
    if not os.path.exists(name):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/INCORTX/INCORTX.github.io/cv-next/ComputerVision/session-09/img/' + name, name)
    return name

cap = cv2.VideoCapture(fetch('street.mp4')); frames = []
while True:
    ok, f = cap.read()
    if not ok: break
    frames.append(f)
T = 60                                                                  # the slides' frame 1060
gray = lambda f: cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
def flow_colour(fl):                                                    # hue = direction, brightness = speed (scaled to this picture)
    mag, ang = cv2.cartToPolar(fl[..., 0], fl[..., 1]); hsv = np.zeros(fl.shape[:2] + (3,), np.uint8)
    hsv[..., 0] = ang * 90 / np.pi; hsv[..., 1] = 255; hsv[..., 2] = np.clip(mag / max(1, np.percentile(mag, 99)) * 255, 0, 255)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
print(len(frames), 'frames of', frames[0].shape[1::-1], '· 23.976 fps → one every 1000 ÷ 23.976 = 41.7 ms')

## A · Optical flow

Brightness stays, position changes: $I_x u + I_y v + I_t = 0$ — one equation per pixel, two unknowns.

In [ ]:
# A1 · Flow on a picture moved by a KNOWN amount (slides 6–7)
u = 2.0     # true motion to the right, px per frame · try 0.5, 12
v = -1.0    # true motion down (negative = up)

a = gray(frames[T]); b = cv2.warpAffine(a, np.float32([[1, 0, u], [0, 1, v]]), a.shape[::-1], borderMode=cv2.BORDER_REFLECT)
fl = cv2.calcOpticalFlowFarneback(a, b, None, 0.5, 4, 21, 3, 5, 1.1, 0)       # 4 pyramid levels, 21-px window
inner = fl[60:-60, 60:-60]                                                  # away from the borders
print(f'true ({u}, {v}) · Farneback median ({np.median(inner[..., 0]):.2f}, {np.median(inner[..., 1]):.2f})')
plt.figure(figsize=(9, 5)); plt.imshow(flow_colour(fl)); plt.axis('off'); plt.title('flow: one colour everywhere = one motion'); plt.show()

**What to notice:** As published: true (2.0, −1.0), Farneback's median (2.00, −1.00) — exact. 0.5 px → (0.50, −1.00); 12 px → (11.99, −1.00): four pyramid levels follow even 12 px. But here the whole picture moves as one — the easy case. In a real street everything moves differently (A2).

In [ ]:
# A2 · Flow between two real frames (slide 6)
gap = 1     # frames apart · try 3, 6

a, b = gray(frames[T]), gray(frames[T + gap])
fl = cv2.calcOpticalFlowFarneback(a, b, None, 0.5, 4, 21, 3, 5, 1.1, 0)
rider = fl[690:850, 404:630]                                                # the cyclist's box at frame T
moving = np.hypot(rider[..., 0], rider[..., 1]) > 5
print(f'{gap} frame(s) = {gap * 41.7:.0f} ms · cyclist median ({np.median(rider[..., 0][moving]):.1f}, {np.median(rider[..., 1][moving]):.1f}) px · building median {np.median(np.hypot(fl[:300, ..., 0], fl[:300, ..., 1])):.2f} px')
plt.figure(figsize=(9, 5)); plt.imshow(flow_colour(fl)); plt.axis('off'); plt.title(f'frame {T} → {T + gap}'); plt.show()

**What to notice:** 1 frame (42 ms): the cyclist moves (30.5, −2.3) px — slide 4's 30 px — while the still building moves 0.03 px. 3 frames (125 ms): the median says (−2.1, −4.7); 6 frames (250 ms): (−3.5, −5.9). He really moved about 90 and 180 px — far beyond what Farneback can follow, so the flow on his box is simply wrong. More frames per second means smaller steps, and flow that works.

In [ ]:
# A3 · u = −Iₜ ÷ Iₓ on one row of brightness (slide 7)
u = 2.0     # true shift in pixels · try 0.5, 8

row = cv2.GaussianBlur(gray(frames[T]).astype(np.float32), (0, 0), 2.0)[718, 606:654]   # the cyclist's tyre → the road
moved = np.interp(np.arange(48) - u, np.arange(48), row)                                  # the row moved right by u
x = 1 + int(np.argmax(row[2:] - row[:-2]))                                                # the steepest pixel
Ix = (row[x + 1] - row[x - 1]) / 2; It = moved[x] - row[x]
print(f'pixel {x}: Ix = ({row[x + 1]:.1f} − {row[x - 1]:.1f}) ÷ 2 = {Ix:.2f} · It = {moved[x]:.1f} − {row[x]:.1f} = {It:.1f} → u = −It ÷ Ix = {-It / Ix:.2f} (true {u})')
plt.plot(row, label='time t'); plt.plot(moved, label='time t + 1'); plt.axvline(x, color='grey', ls='--'); plt.legend(); plt.xlabel('pixel'); plt.ylabel('brightness'); plt.show()

**What to notice:** u = 2: Iₓ = (99.7 − 77.8) ÷ 2 = 10.93, Iₜ = 67.7 − 88.9 = −21.1 → u = 1.93 — the same 1.93 as slide 7 (the clip here is re-encoded, so the brightness values differ a little). u = 0.5 → 0.50 exactly. u = 8 → 3.88: the tangent line only holds while the shift is small next to the ramp (about 8 px wide).

In [ ]:
# A4 · The aperture problem: a bar behind a round hole, normal flow by slide 8's formula
angle = 45      # direction of the bar's normal, degrees (image axes: x right, y down) · try 0, 90

n = np.array([np.cos(np.radians(angle)), np.sin(np.radians(angle))]); true = np.array([2.0, 0.0])
yy, xx = np.mgrid[:240, :240].astype(float)
bar = lambda s: cv2.GaussianBlur(255 * (np.abs((xx - 120 - s[0]) * n[0] + (yy - 120 - s[1]) * n[1]) < 14).astype(np.float32), (0, 0), 4)
a, b = bar((0, 0)), bar(true)                                              # the bar now and one frame later
Iy, Ix = np.gradient(a); It = b - a; g = np.hypot(Ix, Iy)
edge = ((xx - 120) ** 2 + (yy - 120) ** 2 < 70 ** 2) & (g > 0.3 * g.max())  # the bar's edges, inside the hole only
un = -It[edge] / g[edge]                                                    # u⊥ = −It ÷ ‖∇I‖, along ∇I ÷ ‖∇I‖
seen = (un[:, None] * np.stack([Ix[edge], Iy[edge]], 1) / g[edge][:, None]).mean(0)
pred = (true @ n) * n
print(f'true ({true[0]:.1f}, {true[1]:.1f}) · n = ({n[0]:.2f}, {n[1]:.2f}) · (true·n)·n = ({pred[0]:.2f}, {pred[1]:.2f}) · seen through the hole ({seen[0]:.2f}, {seen[1]:.2f})')
show = np.where((xx - 120) ** 2 + (yy - 120) ** 2 < 70 ** 2, a, 128)
plt.imshow(show, cmap='gray', vmin=0, vmax=255); plt.quiver([120], [120], [seen[0]], [-seen[1]], color='r', scale=8); plt.axis('off'); plt.title('red: the motion seen through the hole'); plt.show()

**What to notice:** Normal at 45°, true motion (2, 0): seen through the hole (1.00, 1.00) — slide 8's worked example. Normal at 0° (motion straight across the bar): (1.98, 0.00), almost all of it. Normal at 90° (the bar lies along the motion): (0.00, 0.02) — nothing to see. OpenCV's Farneback returns exactly 0 on a pattern with only one direction, which is why this cell uses slide 8's formula directly.

## B · Lucas–Kanade

Assume a small window moves together: $M\,(u, v)^\top = -(\sum I_xI_t, \sum I_yI_t)^\top$ with $M$ = session 6's matrix of gradients.

In [ ]:
# B1 · Lucas–Kanade in one 15 × 15 window — slide 14's crop, moved by a known (1.0, −0.5) (slides 12–14)
window = 'corner'   # · try 'edge', 'flat'

WIN = {'corner': (55, 60), 'edge': (66, 64), 'flat': (82, 55)}   # slide 14's three windows (x, y)
a = cv2.imread(fetch('lk0.png'), 0).astype(float); b = cv2.imread(fetch('lk1.png'), 0).astype(float)
Ix = np.zeros_like(a); Iy = np.zeros_like(a); Ix[:, 1:-1] = (a[:, 2:] - a[:, :-2]) / 2; Iy[1:-1] = (a[2:] - a[:-2]) / 2; It = b - a
x, y = WIN[window]; w = (slice(y - 7, y + 8), slice(x - 7, x + 8))
M = np.array([[(Ix[w] ** 2).sum(), (Ix[w] * Iy[w]).sum()], [(Ix[w] * Iy[w]).sum(), (Iy[w] ** 2).sum()]])
r = -np.array([(Ix[w] * It[w]).sum(), (Iy[w] * It[w]).sum()])
l2, l1 = np.linalg.eigvalsh(M); Mr = np.round(M) + 0.0                                    # + 0.0 turns −0 into 0
print(f'M = [{Mr[0, 0]:,.0f} {Mr[0, 1]:,.0f} / {Mr[1, 0]:,.0f} {Mr[1, 1]:,.0f}] · λ1 {l1:,.0f} · λ2 {l2:,.0f}')
uv = np.linalg.solve(M, r)
print(f'(u, v) = M⁻¹ (−ΣIxIt, −ΣIyIt) = ({uv[0]:.2f}, {uv[1]:.2f}) · truth (1.0, −0.5)')

**What to notice:** corner: M ≈ [66,856 940 / 940 70,199], λ₁ 70,445 and λ₂ 66,610 → (u, v) = (0.97, −0.41) against the truth (1.0, −0.5) — slide 14's numbers, from the same two images. edge: λ₂ = 73 → (2.13, 1.51), wrong. flat: λ 31 and 29 → (−0.07, −0.17), wrong — the camera noise decides.

In [ ]:
# B2 · Pyramid Lucas–Kanade on the cyclist, 30 px per frame (slide 15)
levels = 3      # pyramid levels above the photo · try 0, 1

a, b = gray(frames[T]), gray(frames[T + 1])
mask = np.zeros_like(a); mask[600:822, 463:605] = 255                      # the rider's body
p0 = cv2.goodFeaturesToTrack(a, 40, 0.01, 8, mask=mask)
p1, st, _ = cv2.calcOpticalFlowPyrLK(a, b, p0, None, winSize=(21, 21), maxLevel=levels, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
d = (p1 - p0).reshape(-1, 2); hit = (st.ravel() == 1) & (np.hypot(d[:, 0] - 30, d[:, 1] + 2) < 3)
print(f'levels {levels}: {hit.sum()} of {len(p0)} corners within 3 px of the truth (30, −2) · median ({np.median(d[:, 0]):.1f}, {np.median(d[:, 1]):.1f})')
crop = cv2.cvtColor(frames[T][560:860, 400:760], cv2.COLOR_BGR2RGB).copy()
for (x0, y0), (dx, dy), h in zip(p0.reshape(-1, 2) - [400, 560], d, hit):
    cv2.arrowedLine(crop, (int(x0), int(y0)), (int(x0 + dx), int(y0 + dy)), (34, 197, 94) if h else (224, 83, 61), 2, tipLength=0.25)
plt.imshow(crop); plt.axis('off'); plt.title('green: right · red: wrong'); plt.show()

**What to notice:** 3 levels: 30 of 40 corners within 3 px of (30, −2), median (29.7, −2.1) — slide 15 had 31 of 40 on the original video. No pyramid: 2 of 40, median (−0.9, −1.6): a 21-px window cannot see a 30-px jump. 1 level (15 px at the top): 3 of 40 — still too far; 3 levels bring it down to 3.75 px.

## C · Tracking

What moved (frame difference) · the same object in the next frame (link boxes by IoU) · a real tracker on the clip.

In [ ]:
# C1 · Frame difference: |Iₜ − Iₜ₋₁| > τ (slide 19)
tau = 25    # · try 10, 60

a = gray(cv2.resize(frames[T - 2], (480, 270), interpolation=cv2.INTER_AREA)).astype(int)
b = gray(cv2.resize(frames[T], (480, 270), interpolation=cv2.INTER_AREA)).astype(int)
moving = np.abs(b - a) > tau
print(f'frames {T - 2} → {T} (83 ms) · |It − It−1| > {tau}: {moving.mean():.1%} of the pixels')
plt.figure(figsize=(9, 5)); plt.imshow(b, cmap='gray'); plt.imshow(np.ma.masked_where(~moving, moving), cmap='autumn', alpha=.9); plt.axis('off'); plt.show()

**What to notice:** τ 25: 4.5 % of the pixels changed in 83 ms — the walkers and the cyclist. τ 10: 7.0 % (noise and video compression start to count); τ 60: 2.1 % (only the strongest edges of what moves).

In [ ]:
# C2 · Link boxes by IoU by hand: highest first, each box once, above a threshold (slides 21–22)
gap = 6             # frames apart · try 1, 12
threshold = 0.3     # · try 0.1, 0.5

yolo = YOLO('yolov8n.pt')
boxes = lambda f: [b for b in yolo(f, classes=[0], conf=0.4, verbose=False)[0].boxes.xyxy.numpy() if b[3] - b[1] >= 60]
def iou(a, b):
    w = max(0, min(a[2], b[2]) - max(a[0], b[0])); h = max(0, min(a[3], b[3]) - max(a[1], b[1])); i = w * h
    return i / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - i)
A, B = boxes(frames[T]), boxes(frames[T + gap])
pairs = sorted(((iou(a, b), i, j) for i, a in enumerate(A) for j, b in enumerate(B)), reverse=True); usedA, usedB, links = set(), set(), []
for val, i, j in pairs:
    if val < threshold: break
    if i in usedA or j in usedB: continue
    usedA.add(i); usedB.add(j); links.append((i, j, val))
print(f'frame {T} → {T + gap} ({gap * 41.7:.0f} ms), threshold {threshold}: {len(links)} linked · {len(B) - len(links)} new IDs · {len(A) - len(links)} lost')
print('lowest kept IoUs:', sorted(round(float(v), 2) for _, _, v in links)[:4])

**What to notice:** 6 frames (250 ms), threshold 0.3: 5 linked, 6 new IDs, 7 lost (slide 22, on the original video: 7 linked). 1 frame (42 ms) at 0.1: 11 linked, 1 new, 1 lost. 12 frames (500 ms) at 0.5: 3 linked, 8 new, 9 lost. Longer gaps and stricter thresholds break the links — exactly slide 22's lesson.

In [ ]:
# C3 · YOLO + a tracker on the whole clip, counting at a line (slide 23)
tracker = 'bytetrack.yaml'      # · try 'botsort.yaml'
LINE = 960                      # a vertical counting line, x in pixels

model = YOLO('yolov8n.pt'); paths = {}; most = 0
for k, f in enumerate(frames):
    r = model.track(f, persist=True, tracker=tracker, classes=[0], verbose=False)[0]
    if r.boxes.id is None: continue
    n = 0
    for b, t in zip(r.boxes.xyxy.numpy(), r.boxes.id.numpy().astype(int)):
        if b[3] - b[1] < 60: continue
        paths.setdefault(t, []).append((b[0] + b[2]) / 2); n += 1
    most = max(most, n)
crossed = [t for t, xs in paths.items() if min(xs) < LINE < max(xs)]
print(f'{tracker}: {len(paths)} IDs in {len(frames) / 23.976:.1f} s · at most {most} people in one frame · {sum(len(x) < 12 for x in paths.values())} IDs lived < 0.5 s · crossed the line: {len(crossed)}')

**What to notice:** ByteTrack: 35 IDs in 3.2 s, never more than 20 people in one frame, 10 IDs lived under half a second, 2 crossed the line. BoT-SORT: 34 IDs, 8 short-lived, 2 crossings. (Slide 23 tracked people, bicycles and cars on the original video: 39 IDs, 3 crossings.) Either way: count crossings, not IDs.

## D · Putting it together

Every stage costs time. Budget per frame = 1000 ÷ fps ms; a stage run every N frames costs ÷ N on average.

In [ ]:
# D1 · Time every stage on THIS machine (slide 29)
size = 640      # YOLO input size · try 320
fps = 24        # · try 30

from ultralytics.trackers.byte_tracker import BYTETracker
from ultralytics.utils import YAML
from types import SimpleNamespace
import ultralytics, statistics
def ms(fn, n=10):
    fn(); out = []
    for k in range(n):
        t0 = time.perf_counter(); fn(); out.append((time.perf_counter() - t0) * 1000)
    return statistics.median(out)
det = YOLO('yolov8n.pt'); res = det(frames[T], imgsz=size, classes=[0], verbose=False)[0]
cfg = YAML.load(ultralytics.__file__.rsplit('/', 1)[0] + '/cfg/trackers/bytetrack.yaml'); cfg['frame_rate'] = 24; trk = BYTETracker(SimpleNamespace(**cfg))
small = [gray(cv2.resize(frames[T + i], (960, 540))) for i in (0, 1)]
cost = {'YOLO': ms(lambda: det(frames[T], imgsz=size, classes=[0], verbose=False)), 'ByteTrack': ms(lambda: trk.update(res.boxes.cpu().numpy(), frames[T])),
        'Farneback 960': ms(lambda: cv2.calcOpticalFlowFarneback(small[0], small[1], None, 0.5, 4, 21, 3, 5, 1.1, 0))}
budget = 1000 / fps; base = cost['YOLO'] + cost['ByteTrack']
print(' · '.join(f'{k} {v:.1f} ms' for k, v in cost.items()))
print(f'budget 1000 ÷ {fps} = {budget:.1f} ms · YOLO + ByteTrack = {base:.1f} ms {"✓" if base <= budget else "✗"} · + flow every frame = {base + cost["Farneback 960"]:.1f} ms {"✓" if base + cost["Farneback 960"] <= budget else "✗"}')

**What to notice:** On the M1 Pro CPU that checked this lab (two checks; timings move by a few ms each run): YOLO at 640 about 30–32 ms + ByteTrack 0.8 ms — fits 24 fps (41.7 ms); add Farneback (about 49 ms) on every frame and it needs about 80 ms — it does not. YOLO at 320: about 12–13 ms, so even 30 fps (33.3 ms) fits. Colab runs on a different machine and prints different numbers — the point is to measure, not to assume.

In [ ]:
# D2 · A whole pipeline — detect + track + count — with the detector run every N frames
every = 1       # run YOLO + the tracker on every N-th frame · try 3, 6

model = YOLO('yolov8n.pt'); paths = {}; t0 = time.perf_counter(); runs = 0
for k in range(0, len(frames), every):
    r = model.track(frames[k], persist=True, tracker='bytetrack.yaml', classes=[0], verbose=False)[0]; runs += 1
    if r.boxes.id is None: continue
    for b, t in zip(r.boxes.xyxy.numpy(), r.boxes.id.numpy().astype(int)):
        if b[3] - b[1] >= 60: paths.setdefault(t, []).append((b[0] + b[2]) / 2)
spent = (time.perf_counter() - t0) * 1000
crossed = sum(min(x) < 960 < max(x) for x in paths.values())
print(f'every {every}: {runs} runs · {spent / len(frames):.1f} ms per video frame on average · {len(paths)} IDs · crossed the line: {crossed}')

**What to notice:** Every frame: 77 runs, about 32–34 ms per video frame, 35 IDs, 2 crossings. Every 3rd: about 11–12 ms per frame — three times faster — but 27 IDs and 0 crossings; every 6th: about 6 ms, 24 IDs, 0 crossings. Between runs the boxes jump too far to link (part C), so the count is lost. A fast answer is worthless if it is wrong: Demo criteria C1 and C2 go together.

---
Back to the slides: [session 9 deck](https://classes.incortx.com/ComputerVision/session-09/slides.html) · exam-style questions: [practice bank](https://classes.incortx.com/ComputerVision/practice/#s9).